In [ ]:
%pwd

'/content'

In [ ]:
#해당 경로에 'jumpit_data', 'data_chroma_db' 폴더 생성
!mkdir jumpit_data data_chroma_db

# jumpit_data에 우리 사용데이터 파일들 넣기, (하위 경로를 추가로 생성하지 말 것)

In [ ]:
!pip install langchain langchain_community langchain-openai langchain_chroma
!pip install unstructured
!python3 -m pip install konlpy

In [ ]:
import os
from glob import glob
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader

In [ ]:
loader = DirectoryLoader('/content/jumpit_data',glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
len(docs)

1

In [ ]:
from konlpy.tag import Okt

okt = Okt()

In [ ]:
def len_okt(text):
    tokens = [token for token in okt.morphs(text)]

    return len(tokens)

def okt_tokenize(text):
    return [token for token in okt.morphs(text)]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " ", ""],
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len_okt
)

In [ ]:
texts = text_splitter.split_documents(docs)

In [ ]:
len(texts)

10

In [ ]:
texts[5]

Document(metadata={'source': 'jumpit_data/Crawling_DataFile_MergePage_json_20250512_test.txt'}, page_content='ECM데스티니 ECM은 고객사 인프라 환경에 따라 커스터마이징 구축이 가능한 문서 중앙화 솔루션","cloudium클라우디움은 하드웨어와 소프트웨어의 과금형 또는 구축과금형 또는 구축형으로 제공되는 SMBSmall and Medium Business 문서 중앙화 솔루션"],"requirements":["관련 경력을 5년 이상 보유한 분","Javascript HTML CSS 기반의 웹 클라이언트 개발이 가능한 분"],"preferential":["기업용 솔루션 개발 경험이 있으신 분","서버 클라이언트 기반 구조에 관련 지식이 있는 분"],"welfare":["직원 중심의 조직문화를 만들어갑니다","유연근무제와 자유로운 연차 사용으로 일과 삶의 균형을 이룰 수 있는 최선의 환경을 제공합니다","휴게실 회의실 카페테리아 노트북 콘도리조트 이용권 도서지원 자격증 응시료 지원 학자금 지원으로 직원들의 자기계발을 존중합니다","사내 저금리1 3천만원 한도 대출 퇴직연금제도 운영으로 직원들의 미래를 지원합니다"],"procedures":["서류전형  1차 면접  2차 면접 필요 시   입사 협의"],"education":"대학졸업2년 이상","closedAt":"2025년 05월 24일","jobCategory":"프론트엔드 개발자","techStack":["JavaScript","CSS 3","HTML5"],"minCareer":5,"maxCareer":15,"jobLocation":["서울 송파구"],"date":"2025년05월12일"},"50479":{"title":"AI 모델 개발자2년 이상","company":"엑셈","pride":["9호선 역세권 기업","택시비지원","맛있는간식냠냠","도서구입비지원"],"mainWork":["데이터 전처리 및 Feature E

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

model_name = 'nlpai-lab/KURE-v1'   # sentence bert - natural language inference(: 자연어 추론)

# model_kwargs = {'device': 'cpu'} # cpu 사용하려면
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True}
hf_embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

<ipython-input-11-91a139d88198>:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/16.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [ ]:
save_directory = './data_chroma_db'

In [ ]:
from langchain_chroma import Chroma
import os
import shutil

print('\n 잠시만 기다려주세요. \n\n')

if os.path.exists(save_directory):
    shutil.rmtree(save_directory)
    print(f'디렉토리 {save_directory}가 삭제되었습니다. \n')

print('문서 벡터화를 시작합니다.')

db = Chroma.from_documents(texts, hf_embedding_model, persist_directory=save_directory)
print('새로운 Chroma데이터베이스가 생성되었습니다.\n')



 잠시만 기다려주세요. 


디렉토리 ./data_chroma_db가 삭제되었습니다. 

문서 벡터화를 시작합니다.
새로운 Chroma데이터베이스가 생성되었습니다.



In [ ]:
# 문서 검색기 정의
retriever = db.as_retriever(
    search_kwargs = {'k': 3}
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 중괄호 안에 들어있는 애들은 변수가 될 것임.
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an assistant for question-ansFEFwering tasks.
            Use the following pieces of retrieved context to answer the question.
            If you don't know the answer, just say that you don't know.

            Answer in Korean.

            #Context:
            {context}
            """,
        ), ("human", "{question}"),
    ]
)

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
os.environ['LANGSMITH_ENDPOINT']= userdata.get('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_PROJECT']= userdata.get('LANGSMITH_PROJECT')
os.environ['LANGSMITH_TRACING']= userdata.get('LANGSMITH_TRACING')
os.environ['LANGSMITH_API_KEY']= userdata.get('LANGSMITH_API_KEY')

In [ ]:
import getpass
import os

if not userdata.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API Key:')

from langchain_openai import ChatOpenAI
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0,
    streaming = True,
    callbacks = [StreamingStdOutCallbackHandler()],
)

In [ ]:
def format_docs(docs):
    return "\n\n".join(document.page_content for document in texts)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

chain = {
    # "context" : retriever,
    "context" : retriever | RunnableLambda(format_docs),
    "question" : RunnablePassthrough()
} | prompt | llm | StrOutputParser()

In [ ]:
question = "프론트 엔지니어 채용공고는 뭐가 있어?"

response = chain.invoke(question)

RateLimitError: Error code: 429 - {'error': {'message': 'Request too large for gpt-4o-mini in organization org-fEK3VXhPCLu9xC9aWKmAwSWO on tokens per min (TPM): Limit 200000, Requested 966396. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [ ]:
question = "난 경력이 1년이야, 내가 제출 할 수 있는 프론트 엔지니어 채용공고와 필요한 기술 스택을 알려줘."
response = chain.invoke(question)

In [ ]:
question = "수평적인 분위기면서도 수직적인 업무 연계가 가능한 공고를 찾고있어."
response = chain.invoke(question)